In [2]:
from tests.indexador_2026 import indexar_problema, indexar_device

import os
from datetime import datetime

problema = 1
computador = "DESKTOP"

teste = False
apenas_conjunto_medio = False
filtrar_algoritmo = False # False para todos
filtrar_config = {
    "GA": [1,2],
    "PSO": [1,2],
    "BO": [1,2]
}
num_runs = 3 if not teste else 2


# diretórios
dadospc = indexar_device(computador)

base_dir = os.path.join(dadospc.base_path, f"Problema {problema}")
local_dir = os.path.join(dadospc.local_path, f"Problema {problema}") # copia ModBase.db pro diretório local

if teste: local_dir = os.path.join(local_dir, "teste")

initimestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
global_log_dir = os.path.join(base_dir, "log", f"rodada_{initimestamp}") if not teste else os.path.join(base_dir, "teste", "log", f"rodada_{initimestamp}")

csv_resultado_path = os.path.join(global_log_dir, f"Resumo Global - Problema {problema}.csv")

# definição dos parâmetros do problema
dadosprob = indexar_problema(problema)

script_name = dadosprob.script_filename
noise = dadosprob.noise
parameters = dadosprob.parameters
target_params = dadosprob.target_params
keys = dadosprob.keys
expected_values = dadosprob.expected_values


# SEUS DOIS (agora três) CONJUNTOS DE HIPERPARÂMETROS
# conjunto 1: explorador (P2 COM 9 PARAM)
# conjunto 2: médio
# conjunto 3: intensificador (P2 COM 6 PARAM)

if teste: # conjunto de teste (rodadas rapidinhas)
    configs_algoritmos = {
        "PSO": [
            {"population_size": len(parameters), "iterations": 4, "w": 0.73, "w_rate": 0.957, "c1": 1.90, "c2": 1.32,
             "init_vel_ratio": 0.06},
            {"population_size": len(parameters), "iterations": 4, "w": 0.6, "w_rate": 0.99, "c1": 2.05, "c2": 2.05,
             "init_vel_ratio": 0.20},
            {"population_size": len(parameters), "iterations": 4, "w": 1.13, "w_rate": 0.964, "c1": 1.15, "c2": 1.43,
             "init_vel_ratio": 0.18}
        ],
        "BO": [
            {"initial_points": len(parameters), "evaluations": 10, "acq_func": 'PI', "xi": 0.1},
            {"initial_points": len(parameters), "evaluations": 10, "acq_func": 'gp_hedge'},
            {"initial_points": len(parameters), "evaluations": 10, "acq_func": 'EI', "xi": 0.003162}
        ],
        "GA": [
            {"population_size": len(parameters), "generations": 4, "elitism_rate": 0.12, "crossover_rate": 0.87,
             "mutation_strength": 0.185},
            {"population_size": len(parameters), "generations": 4, "elitism_rate": 0.10, "crossover_rate": 0.60,
             "mutation_strength": 0.10},
            {"population_size": len(parameters), "generations": 4, "elitism_rate": 0.10, "crossover_rate": 0.75,
             "mutation_strength": 0.25}
        ]
    }

else:
    configs_algoritmos = {
        "PSO": [
            {"population_size": None, "iterations": None, "w": 0.73, "w_rate": 0.957, "c1": 1.90, "c2": 1.32,
             "init_vel_ratio": 0.06},
            {"population_size": None, "iterations": None, "w": 0.6, "w_rate": 0.99, "c1": 2.05, "c2": 2.05,
             "init_vel_ratio": 0.20},
            {"population_size": None, "iterations": None, "w": 1.13, "w_rate": 0.964, "c1": 1.15, "c2": 1.43,
             "init_vel_ratio": 0.18}
        ],
        "BO": [
            {"initial_points": None, "evaluations": None, "acq_func": 'PI', "xi": 0.1},
            {"initial_points": None, "evaluations": None, "acq_func": 'gp_hedge'},
            {"initial_points": None, "evaluations": None, "acq_func": 'EI', "xi": 0.003162}
        ],
        "GA": [
            {"population_size": None, "generations": None, "elitism_rate": 0.12, "crossover_rate": 0.87,
             "mutation_strength": 0.185},
            {"population_size": None, "generations": None, "elitism_rate": 0.10, "crossover_rate": 0.60,
             "mutation_strength": 0.10},
            {"population_size": None, "generations": None, "elitism_rate": 0.10, "crossover_rate": 0.75,
             "mutation_strength": 0.25}
        ]
    }

if apenas_conjunto_medio:
    configs_do_meio = {algoritmo: [configs[1]] for algoritmo, configs in configs_algoritmos.items()}
    configs_algoritmos = configs_do_meio

if filtrar_algoritmo:
    # 1. Descobre quais algoritmos pedidos NÃO estão no dicionário
    faltantes = [algo for algo in filtrar_algoritmo if algo not in configs_algoritmos]

    # 2. Se a lista de faltantes não estiver vazia, para tudo e avisa
    if faltantes:
        raise ValueError(
            f"\n[ERRO CRÍTICO] Execução interrompida."
            f"\nOs seguintes algoritmos não possuem configuração definida: {', '.join(faltantes)}"
        )

    # 3. Se chegou até aqui, é porque todos existem. Filtra com segurança em uma linha!
    configs_algoritmos = {algo: configs_algoritmos[algo] for algo in filtrar_algoritmo}

if filtrar_config:
    for algo in filtrar_config:
        if algo in configs_algoritmos:
            configs_algoritmos[algo] = [configs_algoritmos[algo][i] for i in filtrar_config[algo]]